# Vehicle Price Prediction - EDA & Modeling

This notebook walks through the full data science pipeline:
1. **Data Loading & Inspection**
2. **Exploratory Data Analysis (EDA)**
3. **Data Preprocessing & Feature Engineering**
4. **Model Training & Comparison**
5. **Results & Insights**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

print('Libraries imported successfully.')

---
## 1. Data Loading & Inspection

In [ ]:
df = pd.read_csv('../cleaned_vehicles1.csv')
print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

In [ ]:
print('--- Data Types ---')
print(df.dtypes)
print(f'\n--- Missing Values ---')
print(df.isnull().sum())
print(f'\n--- Unique Values ---')
print(df.nunique())

In [ ]:
df.describe()

---
## 2. Exploratory Data Analysis (EDA)

### 2.1 Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price'], bins=60, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_title('Price Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['price'].mean(), color='red', linestyle='--',
                label=f'Mean: ${df["price"].mean():,.0f}')
axes[0].axvline(df['price'].median(), color='green', linestyle='--',
                label=f'Median: ${df["price"].median():,.0f}')
axes[0].legend()

sns.boxplot(y=df['price'], ax=axes[1], color='#4C72B0', width=0.4)
axes[1].set_title('Price Boxplot', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Price ($)')

plt.tight_layout()
plt.show()

### 2.2 Price by Manufacturer (Top 15)

In [ ]:
top_mfr = df['manufacturer'].value_counts().head(15).index
fig, ax = plt.subplots(figsize=(14, 6))
order = (df[df['manufacturer'].isin(top_mfr)]
         .groupby('manufacturer')['price']
         .median()
         .sort_values(ascending=False)
         .index)
sns.boxplot(data=df[df['manufacturer'].isin(top_mfr)], x='manufacturer', y='price',
            order=order, palette='coolwarm', ax=ax)
ax.set_title('Price by Manufacturer (Top 15)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 2.3 Price vs Odometer

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sample = df.sample(5000, random_state=42)
ax.scatter(sample['odometer'], sample['price'], alpha=0.3, s=10, c='#4C72B0')
ax.set_title('Price vs Odometer', fontsize=14, fontweight='bold')
ax.set_xlabel('Odometer (miles)')
ax.set_ylabel('Price ($)')
plt.tight_layout()
plt.show()

### 2.4 Price by Vehicle Type

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
type_order = df.groupby('type')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='type', y='price', order=type_order, palette='viridis', ax=ax)
ax.set_title('Price by Vehicle Type', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 2.5 Price by Fuel Type

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fuel_order = df.groupby('fuel')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='fuel', y='price', order=fuel_order, palette='Set2', ax=ax)
ax.set_title('Price by Fuel Type', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.6 Correlation Heatmap

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).drop(
    columns=['id', 'price_per_mile'], errors='ignore'
)
fig, ax = plt.subplots(figsize=(10, 8))
corr = numeric_cols.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, square=True)
ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Data Preprocessing & Feature Engineering

In [ ]:
# Drop irrelevant / data-leakage columns
df_clean = df.drop(columns=['id', 'region', 'model', 'price_per_mile'])
print(f'After dropping columns: {df_clean.shape}')

# Extract numeric cylinder count
cylinder_map = {
    '3 cylinders': 3, '4 cylinders': 4, '5 cylinders': 5,
    '6 cylinders': 6, '8 cylinders': 8, '10 cylinders': 10,
    '12 cylinders': 12, 'other': 4, 'Unknown': 4,
}
df_clean['cylinders_num'] = df_clean['cylinders'].map(cylinder_map).fillna(4).astype(int)
df_clean.drop(columns=['cylinders'], inplace=True)
print('Extracted numeric cylinder count')

# Label-encode categorical features
cat_cols = [
    'manufacturer', 'condition', 'fuel', 'title_status',
    'transmission', 'drive', 'type', 'paint_color', 'state',
]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    encoders[col] = le

print(f'Label-encoded {len(cat_cols)} categorical features')
df_clean.head()

In [ ]:
# Feature / Target split
feature_cols = ['year', 'odometer', 'car_age', 'cylinders_num'] + cat_cols
X = df_clean[feature_cols]
y = df_clean['price']

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape[0]:,} samples | Test: {X_test.shape[0]:,} samples')
print(f'Features ({len(feature_cols)}): {feature_cols}')

---
## 4. Model Training & Comparison

In [ ]:
def evaluate(y_true, y_pred, name):
    """Compute and print regression metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name:25s} | MAE: ${mae:>10,.2f} | RMSE: ${rmse:>10,.2f} | R2: {r2:.4f}')
    return {'model': name, 'mae': mae, 'rmse': rmse, 'r2': r2}

In [ ]:
results = []

# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
results.append(evaluate(y_test, lr.predict(X_test), 'Linear Regression'))

# 2. Random Forest
rf = RandomForestRegressor(
    n_estimators=200, max_depth=15, min_samples_split=10,
    min_samples_leaf=5, random_state=42, n_jobs=-1,
)
rf.fit(X_train, y_train)
results.append(evaluate(y_test, rf.predict(X_test), 'Random Forest'))

# 3. XGBoost
xgb = XGBRegressor(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
)
xgb.fit(X_train, y_train)
results.append(evaluate(y_test, xgb.predict(X_test), 'XGBoost'))

In [ ]:
results_df = pd.DataFrame(results)
results_df.style.highlight_min(subset=['mae', 'rmse']).highlight_max(subset=['r2'])

### Model Comparison Chart

In [ ]:
colors = ['#4C72B0', '#55A868', '#C44E52']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric, label in zip(axes, ['mae', 'rmse', 'r2'],
                              ['MAE ($)', 'RMSE ($)', 'R2 Score']):
    bars = ax.bar(results_df['model'], results_df[metric],
                  color=colors, edgecolor='white', width=0.5)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_ylabel(label)
    for bar, val in zip(bars, results_df[metric]):
        fmt = f'${val:,.0f}' if metric != 'r2' else f'{val:.4f}'
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                fmt, ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Feature Importance (XGBoost)

In [ ]:
importances = xgb.feature_importances_
indices = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(importances)), importances[indices],
       color='#C44E52', edgecolor='white', alpha=0.85)
ax.set_xticks(range(len(importances)))
ax.set_xticklabels([feature_cols[i] for i in indices], rotation=45, ha='right')
ax.set_title('Feature Importance (XGBoost)', fontsize=14, fontweight='bold')
ax.set_ylabel('Importance Score')
plt.tight_layout()
plt.show()

### Predicted vs Actual (XGBoost)

In [ ]:
xgb_pred = xgb.predict(X_test)

fig, ax = plt.subplots(figsize=(8, 8))
sample_idx = np.random.RandomState(42).choice(len(y_test), 3000, replace=False)
y_arr = np.array(y_test)

ax.scatter(y_arr[sample_idx], xgb_pred[sample_idx], alpha=0.3, s=10, c='#4C72B0')
lims = [0, max(y_arr.max(), xgb_pred.max()) * 1.05]
ax.plot(lims, lims, '--', color='#C44E52', linewidth=2, label='Perfect Prediction')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Actual Price ($)', fontsize=12)
ax.set_ylabel('Predicted Price ($)', fontsize=12)
ax.set_title('Predicted vs Actual (XGBoost)', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Conclusion

**Key Findings:**
- **XGBoost** is the best-performing model with the highest R2 and lowest error metrics.
- **Year** and **odometer** are the most important features for predicting price.
- The model captures the general pricing trends well, with some variance at extreme price ranges.

**Next Steps:**
- The trained XGBoost model is deployed via Streamlit (`app/streamlit_app.py`).
- Run `streamlit run app/streamlit_app.py` to launch the interactive web app.